# Model results (Fig. 3, Figs. S16-S24, S28 and the all-systems SI figure)

Plotting notebook for the CHELATE model results, four-panel revision (matches Fig. 3).

Run it from this directory (`figures/model/`). Inputs:

- `csv_files/`: the CSV bundle written by `python -m model.test_eval` (retained-system mode,
  i.e. without `--keep_best_pose`) with the retrained models in `model/params/`, plus
  `mbg_sr_vs_training_fraction_predCutoff=VINA-0p8_SMINA-0p85_GNINA-0p8.csv` from
  `scripts/run_sweep.py --pred-cutoffs` (Fig. S28).
- `../../model/params/pred_cutoffs.json`: the prediction cutoff of each docking program,
  chosen on the validation set by `model/select_cutoff.py` (Vina 0.80, smina 0.85,
  GNINA 0.80 Å). Every operating point below uses these.
  To refresh it, copy the evaluation output folder over `csv_files/`.
- `test_system_metals.json`: metals of each test system (from the curated dataset).
- `../../model/data/EC_numbers.csv`: EC numbers of the test PDB entries (Fig. S22).

Figures are written to `figures/` in this directory. The metal-identity probe
(Figs. S25 and S27) is plotted by `../metal_identity/make_metal_identity_figures.py`.
`scripts/paper_numbers.py` prints the paper numbers from the same CSVs (see `NUMBERS.md`).

Baselines are read at the largest prediction cutoff, where all three docking runs
still have retained systems.

The main test curves use the 118-pair common input/label-availability cohort. Per-engine and fixed-/156 accounting is reported separately in `NUMBERS.md` section 7.


In [1]:
import json as _json
import os
from collections import Counter as _Counter
from glob import glob
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.legend import Legend
from matplotlib.lines import Line2D
from scipy import stats

# ---- Prediction cutoff per docking program (validation-selected, model/select_cutoff.py) ----
PRED_CUTOFFS = {
    m: float(v)
    for m, v in _json.loads(
        Path("../../model/params/pred_cutoffs.json").read_text()
    ).items()
    if m in ("VINA", "SMINA", "GNINA")
}
print("Prediction cutoffs (Å):", PRED_CUTOFFS)

# ---- Colors ----
vina_color = "#332288"
smina_color = "#88CCEE"
gnina_color = "#44AA99"
autodock_color = "#d97706"
gpdock_color = "#7c3aed"
metallodock_color = "#CC6677"

# ---- Font / figure sizes ----
fontsize_x_label = 8
fontsize_y_label = 8
fontsize_x_ticks = 8
fontsize_y_ticks = 8
fontsize_legend = 8
fontsize_titles = 8
cm = 1 / 2.54
figsize_two_panel = (16 * cm, 8 * cm)
figsize_three_panel = (16 * cm, 8 * cm)
figsize_four_panel = (16 * cm, 16 * cm)

# ---- Thresholds / method metadata ----
MBG_RMSD_THRESHOLD = 1.0
FULL_RMSD_THRESHOLD = 2.0
EXTRA_BASELINE_METHODS = ["AUTODOCK", "GPDOCK", "METALLODOCK"]
EXTRA_BASELINE_METHOD_COLORS = {
    "AUTODOCK": autodock_color,
    "GPDOCK": gpdock_color,
    "METALLODOCK": metallodock_color,
}
_METHOD_DISPLAY_NAMES = {
    "VINA": "Vina",
    "SMINA": "smina",
    "GNINA": "GNINA",
    "AUTODOCK": "AutoDock GPU",
    "METALLODOCK": "MetalloDock",
    "GPDOCK": "GPDOCK",
}


def _display_method_name(method: str) -> str:
    return _METHOD_DISPLAY_NAMES.get(method, method)


def savefig_padded(fig, path: str, **kwargs):
    kwargs.setdefault("bbox_inches", "tight")
    fig.savefig(path, **kwargs)


def legend_below_axes(fig, axes, handles, labels, pad=0.02, **kwargs):
    """Figure legend whose top sits ``pad`` (figure fraction) below the lowest axes
    decoration (tick labels, x labels, annotations)."""
    renderer = fig.canvas.get_renderer()
    to_fig = fig.transFigure.inverted()
    bottom = min(
        to_fig.transform_bbox(ax.get_tightbbox(renderer)).y0 for ax in np.ravel(axes)
    )
    return fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, bottom - pad),
        frameon=False,
        **kwargs,
    )


def _lighten_color(hex_color: str, factor: float = 0.55) -> tuple:
    r = int(hex_color[1:3], 16) / 255.0
    g = int(hex_color[3:5], 16) / 255.0
    b = int(hex_color[5:7], 16) / 255.0
    return (r + (1 - r) * factor, g + (1 - g) * factor, b + (1 - b) * factor)


def _is_success_rate_metric_name(metric_name: str) -> bool:
    if not isinstance(metric_name, str):
        return False
    return (
        "sr_top" in metric_name
        or "MBG_sr_top" in metric_name
        or "baseline_sr_top" in metric_name
        or "baseline_MBG_sr_top" in metric_name
        or "improvement_top" in metric_name
    )


def _grouped_sr_to_percent(grouped_df: pd.DataFrame) -> pd.DataFrame:
    out = grouped_df.copy()
    for col in out.columns:
        metric_name = col[0] if isinstance(col, tuple) else col
        if _is_success_rate_metric_name(metric_name):
            out[col] = out[col] * 100.0
    return out


def _all_grouped_sr_to_percent(all_grouped: dict) -> dict:
    out = {}
    for method, dock_map in all_grouped.items():
        out[method] = {}
        for dock_key, grouped_df in dock_map.items():
            out[method][dock_key] = _grouped_sr_to_percent(grouped_df)
    return out


def _external_baselines_sr_to_percent(external_baselines: dict | None) -> dict | None:
    if external_baselines is None:
        return None
    out = {}
    for dock_key, method_map in external_baselines.items():
        out[dock_key] = {}
        for method, stats_map in method_map.items():
            out[dock_key][method] = {}
            for key, val in stats_map.items():
                if key in ("coverage", "true_coverage_curve"):
                    out[dock_key][method][key] = val
                    continue
                out[dock_key][method][key] = {}
                for threshold, topk_map in val.items():
                    out[dock_key][method][key][threshold] = {}
                    for top_k, pair in topk_map.items():
                        out[dock_key][method][key][threshold][top_k] = (
                            100.0 * pair[0],
                            100.0 * pair[1],
                        )
    return out


def _unflatten_grouped_df(df_flat: pd.DataFrame) -> pd.DataFrame:
    df = df_flat.copy()
    if "rmsd_cutoff" not in df.columns:
        raise ValueError("Expected 'rmsd_cutoff' in grouped CSV")
    df = df.sort_values("rmsd_cutoff").set_index("rmsd_cutoff")
    tuples = []
    for c in df.columns:
        if "__" in c:
            metric, stat = c.rsplit("__", 1)
            tuples.append((metric, stat))
        else:
            tuples.append((c, "value"))
    df.columns = pd.MultiIndex.from_tuples(tuples)
    return df


def load_plot_repro_inputs_notebook(csv_folder: str):
    grouped_path = os.path.join(csv_folder, "plot_repro_grouped_metrics.csv")
    baselines_path = os.path.join(csv_folder, "plot_repro_external_baselines.csv")

    def _pick(kind):
        return os.path.join(csv_folder, f"plot_repro_prediction_quality_{kind}.csv")

    scatter_path = _pick("scatter")
    corr_path = _pick("corr")
    cutoff_path = _pick("cutoff")

    grouped_all = pd.read_csv(grouped_path)
    all_grouped = {}
    if not grouped_all.empty:
        for method in sorted(grouped_all["method"].unique()):
            all_grouped[method] = {}
            mdf = grouped_all[grouped_all["method"] == method]
            for dock_key in sorted(mdf["dock_key"].unique()):
                ddf = mdf[mdf["dock_key"] == dock_key].copy()
                ddf = ddf.drop(columns=["method", "dock_key"])
                all_grouped[method][dock_key] = _unflatten_grouped_df(ddf)

    baseline_df = (
        pd.read_csv(baselines_path)
        if os.path.exists(baselines_path)
        else pd.DataFrame()
    )
    external_baselines = {}
    if not baseline_df.empty:
        for _, row in baseline_df.iterrows():
            dock_key = row["dock_key"]
            method = row["method"]
            family = row["metric_family"]
            pair = (float(row["mean"]), float(row["std"]))
            external_baselines.setdefault(dock_key, {}).setdefault(method, {})
            if family == "coverage":
                external_baselines[dock_key][method]["coverage"] = pair
                continue
            if family == "true_coverage_curve":
                threshold = float(row["threshold"])
                external_baselines[dock_key][method].setdefault(
                    "true_coverage_curve", {}
                )
                external_baselines[dock_key][method]["true_coverage_curve"][
                    threshold
                ] = pair
                continue
            threshold = float(row["threshold"])
            top_k = int(row["top_k"])
            external_baselines[dock_key][method].setdefault(family, {})
            external_baselines[dock_key][method][family].setdefault(threshold, {})
            external_baselines[dock_key][method][family][threshold][top_k] = pair

    prediction_quality_inputs = {
        "scatter": pd.read_csv(scatter_path)
        if os.path.exists(scatter_path)
        else pd.DataFrame(),
        "corr": pd.read_csv(corr_path) if os.path.exists(corr_path) else pd.DataFrame(),
        "cutoff": pd.read_csv(cutoff_path)
        if os.path.exists(cutoff_path)
        else pd.DataFrame(),
    }
    return all_grouped, external_baselines, prediction_quality_inputs


def plot_four_panel_summary_notebook(
    all_grouped,
    prediction_quality_inputs,
    external_baselines,
    output_folder,
    dock_key,
    show_all_systems=False,
):
    # show_all_systems=True adds the CHELATE MBG success rate over
    # ALL test systems (systems with no retained pose count as failures) as dashed
    # lines in panel c, and writes four_panel_summary_all_systems_{dock_key}.pdf.
    methods = ["VINA", "SMINA", "GNINA"]
    method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}

    all_grouped_pct = _all_grouped_sr_to_percent(all_grouped)
    ext = _external_baselines_sr_to_percent(external_baselines)

    corr_df = prediction_quality_inputs.get("corr", pd.DataFrame())
    cutoff_df = prediction_quality_inputs.get("cutoff", pd.DataFrame())
    if not corr_df.empty:
        corr_df = corr_df[corr_df["dock_key"] == dock_key].copy()
    if not cutoff_df.empty:
        cutoff_df = cutoff_df[cutoff_df["dock_key"] == dock_key].copy()

    # Nested gridspec: each row has its own wspace so the top (1:1) and
    # bottom (2:1) rows do not interfere with each other's column spacing.
    fig = plt.figure(figsize=figsize_four_panel)
    outer = fig.add_gridspec(2, 1, hspace=0.30)
    gs_top = outer[0].subgridspec(1, 2, wspace=0.35)
    gs_bot = outer[1].subgridspec(1, 2, wspace=0.35, width_ratios=[4, 1])
    ax_a = fig.add_subplot(gs_top[0, 0])
    ax_b = fig.add_subplot(gs_top[0, 1])
    ax_c = fig.add_subplot(gs_bot[0, 0])
    ax_d = fig.add_subplot(gs_bot[0, 1])

    # Panel a: per-system correlation histogram
    print(f"\nPanel a \u2013 Average Pearson correlation per system ({dock_key}):")
    bins = np.linspace(-1.0, 1.0, 31)
    for method in methods:
        sub = (
            corr_df[corr_df["method"] == method]
            if not corr_df.empty
            else pd.DataFrame()
        )
        if sub.empty:
            continue
        vals = pd.to_numeric(sub["pearson_r"], errors="coerce")
        vals = vals[np.isfinite(vals)]
        if vals.empty:
            continue
        ax_a.hist(
            vals,
            bins=bins,
            histtype="stepfilled",
            color=method_colors[method],
            alpha=0.2,
            label=f"{_display_method_name(method)}",
        )
        mean_r = float(vals.mean())
        std_r = float(vals.std())
        print(
            f"  {_display_method_name(method)}: mean r = {mean_r:.3f} \u00b1 {std_r:.3f}"
        )
        ax_a.axvline(mean_r, color=method_colors[method], linestyle="--", linewidth=1.2)
    ax_a.legend(
        handles=[Line2D([0], [0], color="grey", linestyle="--", linewidth=1.2)],
        labels=["Mean correlation"],
        loc="upper left",
        fontsize=fontsize_legend,
        frameon=False,
    )
    ax_a.set_xlabel("Pearson Correlation", fontsize=fontsize_x_label)
    ax_a.set_ylabel("Count", fontsize=fontsize_y_label)
    ax_a.set_xlim(-1, 1)
    ax_a.grid(alpha=0.25)
    ax_a.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax_a.tick_params(axis="y", labelsize=fontsize_y_ticks)

    # Panel b: correctly retained / correctly discarded systems
    cutoff_grouped = None
    if not cutoff_df.empty:
        cutoff_grouped = cutoff_df.groupby(
            ["method", "rmsd_cutoff"], as_index=False
        ).agg(
            frac_retained_with_success_mean=("frac_retained_with_success", "mean"),
            frac_retained_with_success_std=("frac_retained_with_success", "std"),
            frac_correctly_discarded_mean=("frac_correctly_discarded", "mean"),
            frac_correctly_discarded_std=("frac_correctly_discarded", "std"),
            theoretical_sr_top1_mean=("theoretical_sr_top1", "mean"),
            theoretical_sr_top1_std=("theoretical_sr_top1", "std"),
        )
        for c in [
            "frac_retained_with_success_std",
            "frac_correctly_discarded_std",
            "theoretical_sr_top1_std",
        ]:
            cutoff_grouped[c] = cutoff_grouped[c].fillna(0.0)

    if cutoff_grouped is not None:
        for method in methods:
            subm = cutoff_grouped[cutoff_grouped["method"] == method].sort_values(
                "rmsd_cutoff"
            )
            if subm.empty:
                continue
            ax_b.errorbar(
                subm["rmsd_cutoff"],
                subm["frac_retained_with_success_mean"],
                yerr=subm["frac_retained_with_success_std"],
                marker="o",
                markersize=4,
                linestyle="-",
                capsize=3,
                color=method_colors[method],
            )
            ax_b.errorbar(
                subm["rmsd_cutoff"],
                subm["frac_correctly_discarded_mean"],
                yerr=subm["frac_correctly_discarded_std"],
                marker="s",
                markersize=4,
                linestyle="-",
                capsize=3,
                color=method_colors[method],
            )

    for method in methods:
        ax_b.axvline(
            PRED_CUTOFFS[method],
            color=method_colors[method],
            linestyle="--",
            linewidth=1.5,
        )

    ax_b.set_xlabel("Predicted MBG RMSD Cutoff (\u00c5)", fontsize=fontsize_x_label)
    ax_b.set_ylabel("Fraction of systems", fontsize=fontsize_y_label)
    ax_b.set_ylim(0, 1)
    ax_b.legend(
        handles=[
            Line2D(
                [0],
                [0],
                color="grey",
                marker="o",
                linestyle="-",
                label="Kept, good",
                markersize=4,
            ),
            Line2D(
                [0],
                [0],
                color="grey",
                marker="s",
                linestyle="-",
                label="Discarded, bad",
                markersize=4,
            ),
            Line2D(
                [0],
                [0],
                color="grey",
                linestyle="--",
                linewidth=2.0,
                label="Sel. cutoffs",
            ),
        ],
        fontsize=fontsize_legend,
        frameon=False,
        loc="center right",
        bbox_to_anchor=(1.0, 0.42),
        borderaxespad=0.2,
        handlelength=1.6,
    )
    ax_b.grid(alpha=0.25)
    ax_b.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax_b.tick_params(axis="y", labelsize=fontsize_y_ticks)

    # Panel c: MBG SR
    x_ref = None
    for method in methods:
        grp = all_grouped_pct[method][dock_key]
        x = grp.index.values
        x_ref = x
        ax_c.errorbar(
            x,
            grp[(f"MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "mean")],
            yerr=grp[(f"MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "std")],
            marker="o",
            markersize=4,
            linestyle="-",
            capsize=3,
            color=method_colors[method],
            label=f"{_display_method_name(method)}",
        )

    _all_sys_col = f"MBG_sr_all_systems_top1_RMSD={MBG_RMSD_THRESHOLD:g}"
    if show_all_systems:
        for method in methods:
            grp = all_grouped_pct[method][dock_key]
            if (_all_sys_col, "mean") not in grp.columns:
                continue
            # The all-systems rate is a success rate but its name does not match
            # _is_success_rate_metric_name, so convert it to percent here.
            ax_c.errorbar(
                grp.index.values,
                100.0 * grp[(_all_sys_col, "mean")],
                yerr=100.0 * grp[(_all_sys_col, "std")],
                marker="^",
                markersize=4,
                linestyle="--",
                capsize=3,
                color=method_colors[method],
            )

    if x_ref is not None:
        for b_method in ["VINA", "SMINA", "GNINA"]:
            grp_b = all_grouped_pct[b_method][dock_key]
            b_mbg = grp_b[
                (f"baseline_MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "mean")
            ].iloc[-1]
            b_mbg_std = grp_b[
                (f"baseline_MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "std")
            ].iloc[-1]
            ax_c.axhline(
                b_mbg,
                color=method_colors[b_method],
                linestyle="-.",
                linewidth=1.2,
                label=f"{_display_method_name(b_method)} baseline",
            )
            ax_c.fill_between(
                x_ref,
                b_mbg - b_mbg_std,
                b_mbg + b_mbg_std,
                color=method_colors[b_method],
                alpha=0.10,
            )
        for b_method in EXTRA_BASELINE_METHODS:
            mbg_entry = (
                ext.get(dock_key, {})
                .get(b_method, {})
                .get("mbg", {})
                .get(MBG_RMSD_THRESHOLD, {})
                .get(1)
                if ext
                else None
            )
            if mbg_entry is not None:
                ax_c.axhline(
                    mbg_entry[0],
                    color=EXTRA_BASELINE_METHOD_COLORS[b_method],
                    linestyle="-.",
                    linewidth=1.2,
                    label=f"{_display_method_name(b_method)} baseline",
                )
                ax_c.fill_between(
                    x_ref,
                    mbg_entry[0] - mbg_entry[1],
                    mbg_entry[0] + mbg_entry[1],
                    color=EXTRA_BASELINE_METHOD_COLORS[b_method],
                    alpha=0.10,
                )

    if cutoff_grouped is not None:
        for method in methods:
            subm = cutoff_grouped[cutoff_grouped["method"] == method].sort_values(
                "rmsd_cutoff"
            )
            if subm.empty:
                continue
            ax_c.errorbar(
                subm["rmsd_cutoff"],
                100.0 * subm["theoretical_sr_top1_mean"],
                yerr=100.0 * subm["theoretical_sr_top1_std"],
                marker="",
                linestyle="-",
                capsize=3,
                color=_lighten_color(method_colors[method]),
                label=f"{_display_method_name(method)} theoretical limit",
            )

    # Print MBG SR values at each program's selected cutoff for CHELATE models
    print(f"\nPanel c \u2013 MBG Success Rate at the selected cutoffs ({dock_key}):")
    for method in methods:
        grp = all_grouped_pct[method][dock_key]
        idx_pos = int(np.abs(grp.index.values - PRED_CUTOFFS[method]).argmin())
        cutoff_val = float(grp.index[idx_pos])
        mean_val = float(
            grp.iloc[idx_pos][(f"MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "mean")]
        )
        std_val = float(
            grp.iloc[idx_pos][(f"MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "std")]
        )
        print(
            f"  {_display_method_name(method)}: "
            f"MBG SR = {mean_val:.1f} \u00b1 {std_val:.1f}% "
            f"(at cutoff = {cutoff_val:.2f} \u00c5)"
        )

    print(f"\nPanel c \u2013 MBG Success Rate for baseline methods ({dock_key}):")
    for b_method in methods:
        grp_b = all_grouped_pct[b_method][dock_key]
        b_mbg = float(
            grp_b[(f"baseline_MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "mean")].iloc[
                -1
            ]
        )
        b_mbg_std = float(
            grp_b[(f"baseline_MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}", "std")].iloc[-1]
        )
        print(
            f"  {_display_method_name(b_method)} baseline: "
            f"MBG SR = {b_mbg:.1f} \u00b1 {b_mbg_std:.1f}%"
        )

    for method in methods:
        ax_c.axvline(
            PRED_CUTOFFS[method],
            color=method_colors[method],
            linestyle="--",
            linewidth=1.5,
        )
    ax_c.set_xlabel("Predicted MBG RMSD Cutoff (\u00c5)", fontsize=fontsize_x_label)
    ax_c.set_ylabel("MBG Success Rate (%)", fontsize=fontsize_y_label)
    ax_c.set_ylim(0, 80 if dock_key == "crossdocked" else 100)
    ax_c.grid(alpha=0.25)
    ax_c.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax_c.tick_params(axis="y", labelsize=fontsize_y_ticks)

    cutoff_x_values = all_grouped_pct[methods[0]][dock_key].index.values
    for ax in [ax_b, ax_c]:
        ax.set_xlim(float(np.min(cutoff_x_values)), float(np.max(cutoff_x_values)))

    # Collect legend handles from panels a and c for panel d
    # (ax_a.legend() was already set above; get_legend_handles_labels still returns
    #  only artists with labels, so "Mean correlation" won't bleed into panel d)
    handles, labels = [], []
    for ax in [ax_c, ax_a]:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
    by_label = {}
    for h, l in zip(handles, labels):
        if l not in by_label:
            by_label[l] = h

    chelate_labels = [
        l
        for l in by_label
        if not l.endswith(" baseline") and not l.endswith(" theoretical limit")
    ]
    baseline_labels = [l for l in by_label if l.endswith(" baseline")]
    theoretical_labels = [l for l in by_label if l.endswith(" theoretical limit")]

    chelate_handles = [by_label[l] for l in chelate_labels]
    baseline_handles = [by_label[l] for l in baseline_labels]
    theoretical_handles = [by_label[l] for l in theoretical_labels]

    baseline_short_labels = [l.replace(" baseline", "") for l in baseline_labels]
    theoretical_short_labels = [
        l.replace(" theoretical limit", "") for l in theoretical_labels
    ]

    # Panel d: legends
    ax_d.set_axis_off()

    if show_all_systems:
        # Line-style key for the two CHELATE success-rate definitions, inside panel c.
        ax_c.add_artist(
            Legend(
                ax_c,
                [
                    Line2D(
                        [0], [0], color="grey", marker="o", markersize=4, linestyle="-"
                    ),
                    Line2D(
                        [0], [0], color="grey", marker="^", markersize=4, linestyle="--"
                    ),
                ],
                ["CHELATE, retained systems", "CHELATE, all systems"],
                loc="upper right",
                fontsize=fontsize_legend,
                frameon=False,
            )
        )
    leg1 = Legend(
        ax_d,
        chelate_handles,
        chelate_labels,
        loc="upper right",
        bbox_to_anchor=(0.8, 1.0),
        bbox_transform=ax_d.transAxes,
        ncol=1,
        fontsize=fontsize_legend,
        frameon=False,
        title="CHELATE trained on:",
        title_fontsize=fontsize_legend,
    )
    ax_d.add_artist(leg1)

    leg2 = Legend(
        ax_d,
        baseline_handles,
        baseline_short_labels,
        loc="upper right",
        bbox_to_anchor=(0.8, 0.68),
        bbox_transform=ax_d.transAxes,
        ncol=1,
        fontsize=fontsize_legend,
        frameon=False,
        title="Baselines",
        title_fontsize=fontsize_legend,
    )
    ax_d.add_artist(leg2)

    leg3 = Legend(
        ax_d,
        theoretical_handles,
        theoretical_short_labels,
        loc="upper right",
        bbox_to_anchor=(0.8, 0.15),
        bbox_transform=ax_d.transAxes,
        ncol=1,
        fontsize=fontsize_legend,
        frameon=False,
        title="Theoretical limits",
        title_fontsize=fontsize_legend,
    )
    ax_d.add_artist(leg3)

    for ax, lbl, ttl in zip(
        [ax_a, ax_b, ax_c],
        ["a", "b", "c"],
        [
            "Pearson correlations per system",
            "Correctly retained/discarded systems",
            "MBG Success Rate",
        ],
    ):
        ax.set_title(f"({lbl}) {ttl}".rstrip(), fontsize=fontsize_titles, loc="left")

    fig.tight_layout()
    # Same gap above "Theoretical limits" as above "Baselines".
    renderer = fig.canvas.get_renderer()
    to_axes = ax_d.transAxes.inverted()
    box1, box2, box3 = (
        to_axes.transform_bbox(leg.get_window_extent(renderer))
        for leg in (leg1, leg2, leg3)
    )
    leg3.set_bbox_to_anchor(
        (0.8, box2.y0 - (box1.y0 - box2.y1) + (0.15 - box3.y1)),
        transform=ax_d.transAxes,
    )
    savefig_padded(
        fig,
        os.path.join(
            output_folder,
            f"four_panel_summary_{'all_systems_' if show_all_systems else ''}{dock_key}.pdf",
        ),
    )
    plt.show()


manifest_path = sorted(
    glob("csv_files/plot_repro_manifest.json"),
    key=os.path.getmtime,
    reverse=True,
)[0]
csv_folder = str(Path(manifest_path).parent)
output_folder = "figures/"
os.makedirs(output_folder, exist_ok=True)
all_grouped, external_baselines, prediction_quality_inputs = (
    load_plot_repro_inputs_notebook(csv_folder)
)

Prediction cutoffs (Å): {'VINA': 0.8, 'SMINA': 0.85, 'GNINA': 0.8}


In [2]:
def plot_clean_outputs(all_grouped, external_baselines, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    all_grouped = _all_grouped_sr_to_percent(all_grouped)
    external_baselines = _external_baselines_sr_to_percent(external_baselines)

    methods = ["VINA", "SMINA", "GNINA"]
    dock_keys = ["redocked", "crossdocked"]
    dock_labels = {"redocked": "Re-docked", "crossdocked": "Cross-docked"}
    method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}

    def _plot_with_baseline(ax, x, y, yerr, b_mean, b_std, color, label):
        ax.errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            linestyle="-",
            capsize=3,
            color=color,
            label=f"{label}",
        )
        ax.axhline(
            b_mean,
            color=color,
            linestyle="-.",
            linewidth=1.2,
            label=f"{label} baseline",
        )
        ax.fill_between(x, b_mean - b_std, b_mean + b_std, color=color, alpha=0.12)

    def _finalize_two_panel(fig, axes):
        handles, labels = [], []
        for ax in axes:
            h, l = ax.get_legend_handles_labels()
            handles.extend(h)
            labels.extend(l)
        by_label = {}
        for h, l in zip(handles, labels):
            if l not in by_label:
                by_label[l] = h
        chelate_labels = [l for l in by_label if not l.endswith(" baseline")]
        baseline_labels = [l for l in by_label if l.endswith(" baseline")]
        chelate_handles = [by_label[l] for l in chelate_labels]
        baseline_handles = [by_label[l] for l in baseline_labels]
        baseline_short_labels = [l.replace(" baseline", "") for l in baseline_labels]
        fig.tight_layout()
        fig.legend(
            chelate_handles,
            chelate_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.04),
            ncol=3,
            fontsize=fontsize_legend,
            frameon=False,
            title="CHELATE trained on:",
            title_fontsize=fontsize_legend,
        )
        fig.legend(
            baseline_handles,
            baseline_short_labels,
            loc="upper center",
            bbox_to_anchor=(0.5, -0.06),
            ncol=max(1, len(baseline_handles)),
            fontsize=fontsize_legend,
            frameon=False,
            title="Baselines",
            title_fontsize=fontsize_legend,
        )

    # whole-molecule success rate
    fig_full, axes_full = plt.subplots(1, 2, figsize=figsize_two_panel)
    for ax, dock in zip(axes_full, dock_keys):
        x_ref = None
        for method in methods:
            grp = all_grouped[method][dock]
            x = grp.index.values
            x_ref = x
            _plot_with_baseline(
                ax,
                x,
                grp[(f"sr_top1_RMSD={FULL_RMSD_THRESHOLD:g}", "mean")],
                grp[(f"sr_top1_RMSD={FULL_RMSD_THRESHOLD:g}", "std")],
                grp[(f"baseline_sr_top1_RMSD={FULL_RMSD_THRESHOLD:g}", "mean")].iloc[
                    -1
                ],
                grp[(f"baseline_sr_top1_RMSD={FULL_RMSD_THRESHOLD:g}", "std")].iloc[-1],
                method_colors[method],
                _display_method_name(method),
            )
        if x_ref is not None:
            for b_method in EXTRA_BASELINE_METHODS:
                b_color = EXTRA_BASELINE_METHOD_COLORS[b_method]
                e = (
                    external_baselines.get(dock, {})
                    .get(b_method, {})
                    .get("rmsd", {})
                    .get(FULL_RMSD_THRESHOLD, {})
                    .get(1)
                )
                if e is not None:
                    ax.axhline(
                        e[0],
                        color=b_color,
                        linestyle="-.",
                        linewidth=1.2,
                        label=f"{_display_method_name(b_method)} baseline",
                    )
                    ax.fill_between(
                        x_ref, e[0] - e[1], e[0] + e[1], color=b_color, alpha=0.12
                    )
            ax.set_xlim(float(np.min(x_ref)), float(np.max(x_ref)))
        ax.set_xlabel("Predicted MBG RMSD Cutoff (Å)", fontsize=fontsize_x_label)
        ax.set_title(dock_labels[dock], fontsize=fontsize_titles)
        ax.grid(alpha=0.25)
        ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
        ax.tick_params(axis="y", labelsize=fontsize_y_ticks)
    axes_full[0].set_ylabel(
        "Whole-Molecule Success Rate (%)",
        fontsize=fontsize_y_label,
    )
    _finalize_two_panel(fig_full, axes_full)
    savefig_padded(fig_full, os.path.join(output_folder, "clean_full_success_rate.pdf"))
    plt.show()

    # binding affinity Pearson R
    fig_aff, axes_aff = plt.subplots(1, 2, figsize=figsize_two_panel)
    for ax, dock in zip(axes_aff, dock_keys):
        x_ref = None
        for method in methods:
            grp = all_grouped[method][dock]
            x = grp.index.values
            x_ref = x
            _plot_with_baseline(
                ax,
                x,
                grp[("affinity_Pearson_R", "mean")],
                grp[("affinity_Pearson_R", "std")],
                grp[("baseline_affinity_Pearson_R", "mean")].iloc[-1],
                grp[("baseline_affinity_Pearson_R", "std")].iloc[-1],
                method_colors[method],
                _display_method_name(method),
            )
        if x_ref is not None:
            ax.set_xlim(float(np.min(x_ref)), float(np.max(x_ref)))
        ax.set_xlabel("Predicted MBG RMSD Cutoff (Å)", fontsize=fontsize_x_label)
        ax.set_title(dock_labels[dock], fontsize=fontsize_titles)
        ax.grid(alpha=0.25)
        ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
        ax.tick_params(axis="y", labelsize=fontsize_y_ticks)
    axes_aff[0].set_ylabel("Binding Affinity Pearson R", fontsize=fontsize_y_label)
    _finalize_two_panel(fig_aff, axes_aff)
    savefig_padded(fig_aff, os.path.join(output_folder, "clean_affinity_pearson_r.pdf"))
    plt.show()


plot_clean_outputs(
    all_grouped=all_grouped,
    external_baselines=external_baselines,
    output_folder=output_folder,
)

In [3]:
plot_four_panel_summary_notebook(
    all_grouped=all_grouped,
    prediction_quality_inputs=prediction_quality_inputs,
    external_baselines=external_baselines,
    output_folder=output_folder,
    dock_key="redocked",
)


Panel a – Average Pearson correlation per system (redocked):
  Vina: mean r = 0.414 ± 0.232
  smina: mean r = 0.435 ± 0.246
  GNINA: mean r = 0.491 ± 0.237

Panel c – MBG Success Rate at the selected cutoffs (redocked):
  Vina: MBG SR = 42.5 ± 1.7% (at cutoff = 0.80 Å)
  smina: MBG SR = 45.5 ± 1.2% (at cutoff = 0.85 Å)
  GNINA: MBG SR = 49.3 ± 1.1% (at cutoff = 0.80 Å)

Panel c – MBG Success Rate for baseline methods (redocked):
  Vina baseline: MBG SR = 31.6 ± 1.3%
  smina baseline: MBG SR = 32.2 ± 1.5%
  GNINA baseline: MBG SR = 33.9 ± 1.7%


In [4]:
plot_four_panel_summary_notebook(
    all_grouped=all_grouped,
    prediction_quality_inputs=prediction_quality_inputs,
    external_baselines=external_baselines,
    output_folder=output_folder,
    dock_key="crossdocked",
)


Panel a – Average Pearson correlation per system (crossdocked):
  Vina: mean r = 0.401 ± 0.238
  smina: mean r = 0.430 ± 0.233
  GNINA: mean r = 0.489 ± 0.241

Panel c – MBG Success Rate at the selected cutoffs (crossdocked):
  Vina: MBG SR = 32.2 ± 2.7% (at cutoff = 0.80 Å)
  smina: MBG SR = 34.0 ± 1.6% (at cutoff = 0.85 Å)
  GNINA: MBG SR = 40.2 ± 1.0% (at cutoff = 0.80 Å)

Panel c – MBG Success Rate for baseline methods (crossdocked):
  Vina baseline: MBG SR = 18.1 ± 0.5%
  smina baseline: MBG SR = 18.4 ± 1.3%
  GNINA baseline: MBG SR = 18.1 ± 2.1%


In [5]:
# Summary figure with the all-systems MBG success rate (crossdocked)
plot_four_panel_summary_notebook(
    all_grouped=all_grouped,
    prediction_quality_inputs=prediction_quality_inputs,
    external_baselines=external_baselines,
    output_folder=output_folder,
    dock_key="crossdocked",
    show_all_systems=True,
)


Panel a – Average Pearson correlation per system (crossdocked):
  Vina: mean r = 0.401 ± 0.238
  smina: mean r = 0.430 ± 0.233
  GNINA: mean r = 0.489 ± 0.241

Panel c – MBG Success Rate at the selected cutoffs (crossdocked):
  Vina: MBG SR = 32.2 ± 2.7% (at cutoff = 0.80 Å)
  smina: MBG SR = 34.0 ± 1.6% (at cutoff = 0.85 Å)
  GNINA: MBG SR = 40.2 ± 1.0% (at cutoff = 0.80 Å)

Panel c – MBG Success Rate for baseline methods (crossdocked):
  Vina baseline: MBG SR = 18.1 ± 0.5%
  smina baseline: MBG SR = 18.4 ± 1.3%
  GNINA baseline: MBG SR = 18.1 ± 2.1%


In [6]:
# Summary figure with the all-systems MBG success rate (redocked)
plot_four_panel_summary_notebook(
    all_grouped=all_grouped,
    prediction_quality_inputs=prediction_quality_inputs,
    external_baselines=external_baselines,
    output_folder=output_folder,
    dock_key="redocked",
    show_all_systems=True,
)


Panel a – Average Pearson correlation per system (redocked):
  Vina: mean r = 0.414 ± 0.232
  smina: mean r = 0.435 ± 0.246
  GNINA: mean r = 0.491 ± 0.237

Panel c – MBG Success Rate at the selected cutoffs (redocked):
  Vina: MBG SR = 42.5 ± 1.7% (at cutoff = 0.80 Å)
  smina: MBG SR = 45.5 ± 1.2% (at cutoff = 0.85 Å)
  GNINA: MBG SR = 49.3 ± 1.1% (at cutoff = 0.80 Å)

Panel c – MBG Success Rate for baseline methods (redocked):
  Vina baseline: MBG SR = 31.6 ± 1.3%
  smina baseline: MBG SR = 32.2 ± 1.5%
  GNINA baseline: MBG SR = 33.9 ± 1.7%


In [7]:
# ── Configuration ─────────────────────────────────────────────────────────────
_SUCCESS_THR = 1.0
_DOCK_KEY = "crossdocked"
_METHODS = ["VINA", "SMINA", "GNINA"]
_METHOD_COLORS = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}

EC_csv_path = "../../model/data/EC_numbers.csv"
EC_df = pd.read_csv(EC_csv_path, encoding="utf-8-sig")


def has_ec_number(pdb_id):
    row = EC_df[EC_df["PDB ID"] == pdb_id]
    if not row.empty:
        return pd.notna(row.iloc[0]["EC raw text"]) and row.iloc[0]["EC raw text"] != ""
    return False


# ── Load per-system metal information ─────────────────────────────────────────
_METAL_MAP_PATH = "test_system_metals.json"
with open(_METAL_MAP_PATH) as _fh:
    _ligand_metal_map = _json.load(_fh)

# ── Build per-system metadata table ───────────────────────────────────────────
_scatter = prediction_quality_inputs["scatter"].copy()
_sys_keys = _scatter["system_id__ligand_id"].unique()

_rows = []
for _key in _sys_keys:
    _sid = _key.rsplit("__", 1)[0]
    _pdb = _sid.split("__")[0]
    _metals = frozenset(m[0].upper() for m in _ligand_metal_map.get(_sid, []))
    _rows.append(
        {
            "system_id__ligand_id": _key,
            "entry_pdb_id": _pdb,
            "is_enzyme": has_ec_number(_pdb),
            "metals": _metals,
        }
    )
_sys_meta = pd.DataFrame(_rows)

# ── Identify top-5 metals in the test set ─────────────────────────────────────
_metal_counts = _Counter()
for _ms in _sys_meta["metals"]:
    _metal_counts.update(_ms)
_top5_metals = [m for m, _ in _metal_counts.most_common(5)]
# Ni and Co tie at 10 test systems each; pin the set shown in Fig. S22.
_top5_metals = ["ZN", "MG", "FE", "CA", "NI"]
print("Top-5 metals in test set:", _top5_metals)
print("Metal counts:", dict(_metal_counts.most_common(6)))


def _subset_sr(
    scatter_df,
    sys_meta,
    mask_series,
    method,
    dock_key,
    pred_cutoff=None,
    success_thr=_SUCCESS_THR,
):
    if pred_cutoff is None:
        pred_cutoff = PRED_CUTOFFS[method]
    kept_ids = set(sys_meta.loc[mask_series.values, "system_id__ligand_id"])

    df = scatter_df[
        (scatter_df["method"] == method)
        & (scatter_df["dock_key"] == dock_key)
        & (scatter_df["system_id__ligand_id"].isin(kept_ids))
    ].copy()

    base_list, chel_list = [], []
    base_ns, chel_ns = [], []
    for rep in sorted(df["rep_id"].unique()):
        rdf = df[df["rep_id"] == rep]

        top1 = rdf[rdf["pose"] == 1]
        base_ns.append(int(top1["system_id__ligand_id"].nunique()))
        base_list.append(
            (top1["true_mbg_rmsd"] <= success_thr).mean() * 100.0 if len(top1) else 0.0
        )

        filt = rdf[rdf["pred"] <= pred_cutoff]
        chel_ns.append(int(filt["system_id__ligand_id"].nunique()))
        if filt.empty:
            chel_list.append(0.0)
        else:
            best = filt.sort_values("pose").groupby("system_id__ligand_id").first()
            chel_list.append((best["true_mbg_rmsd"] <= success_thr).mean() * 100.0)

    bm = float(np.mean(base_list))
    bs = float(np.std(base_list, ddof=1)) if len(base_list) > 1 else 0.0
    cm = float(np.mean(chel_list))
    cs = float(np.std(chel_list, ddof=1)) if len(chel_list) > 1 else 0.0
    return bm, bs, cm, cs, base_ns, chel_ns


def _add_n_label(ax, x, height, yerr, ns, color, fontsize=5, single=False):
    """Place a rotated per-rep n label just above the error bar cap."""
    y = height + yerr + 1.5
    label = str(ns[0]) if single else "/".join(str(n) for n in ns)
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="bottom",
        fontsize=fontsize,
        color=color,
        rotation=90,
    )


_panel_letters = ["a", "b", "c", "d", "e", "f"]
_metal_labels = {
    "ZN": "Zn",
    "MG": "Mg",
    "CA": "Ca",
    "FE": "Fe",
    "CO": "Co",
    "MN": "Mn",
    "NI": "Ni",
    "CU": "Cu",
}

_panels = []
_panels.append(
    {
        "title": "(a) Enzyme vs. non-enzyme",
        "groups": [
            ("Enzymes", _sys_meta["is_enzyme"] == True),
            ("Non-enzymes", _sys_meta["is_enzyme"] == False),
        ],
    }
)
for _metal in _top5_metals:
    _lbl = _metal_labels.get(_metal, _metal.capitalize())
    _idx = len(_panels)
    _panels.append(
        {
            "title": f"({_panel_letters[_idx]}) {_lbl} systems",
            "groups": [
                (_lbl, _sys_meta["metals"].apply(lambda s, m=_metal: m in s)),
            ],
        }
    )

# ── Pre-compute all success rates ─────────────────────────────────────────────
_panel_results = []
for _pspec in _panels:
    _grp_res = []
    for _gname, _gmask in _pspec["groups"]:
        _mres = {}
        for _m in _METHODS:
            _mres[_m] = _subset_sr(_scatter, _sys_meta, _gmask, _m, _DOCK_KEY)
        _grp_res.append((_gname, _mres))
    _panel_results.append(_grp_res)

# ── Plot ───────────────────────────────────────────────────────────────────────
_BAR_W = 0.12
_PAIR_G = 0.04
_OFFSETS = []
_pair_w = 2 * _BAR_W + _PAIR_G
_total = len(_METHODS) * _pair_w - _PAIR_G
_start = -_total / 2 + _BAR_W / 2
for _mi in range(len(_METHODS)):
    _base_off = _start + _mi * _pair_w
    _chel_off = _base_off + _BAR_W
    _OFFSETS.extend([_base_off, _chel_off])

_METAL_BAR_W = 0.20

fig_aa, axes_aa = plt.subplots(2, 3, figsize=(16 * cm, 12 * cm))
_axs = axes_aa.ravel()

for _pi, (_pspec, _grp_res) in enumerate(zip(_panels, _panel_results)):
    ax = _axs[_pi]

    if _pi == 0:
        _n_grps = len(_grp_res)
        _x_cents = np.arange(_n_grps, dtype=float)

        for _gi, (_gname, _mres) in enumerate(_grp_res):
            _bar_i = 0
            for _mi, _m in enumerate(_METHODS):
                _col = _METHOD_COLORS[_m]
                _bm, _bs, _cm, _cs, _bns, _cns = _mres[_m]
                _lbl_base = f"{_display_method_name(_m)} baseline" if _gi == 0 else ""
                _lbl_chel = f"CHELATE ({_display_method_name(_m)})" if _gi == 0 else ""

                _bx = _x_cents[_gi] + _OFFSETS[_bar_i]
                _cx = _x_cents[_gi] + _OFFSETS[_bar_i + 1]

                ax.bar(
                    _bx,
                    _bm,
                    _BAR_W,
                    yerr=_bs,
                    capsize=2,
                    color=_col,
                    alpha=0.30,
                    label=_lbl_base,
                    error_kw={"elinewidth": 0.8},
                )
                _add_n_label(ax, _bx, _bm, _bs, _bns, _col, single=True)

                ax.bar(
                    _cx,
                    _cm,
                    _BAR_W,
                    yerr=_cs,
                    capsize=2,
                    color=_col,
                    alpha=1.0,
                    label=_lbl_chel,
                    error_kw={"elinewidth": 0.8},
                )
                _add_n_label(ax, _cx, _cm, _cs, _cns, _col)

                _bar_i += 2

        _xpad = 0.4
        ax.set_xlim(_x_cents[0] - _xpad, _x_cents[-1] + _xpad)
        ax.set_xticks(_x_cents)
        ax.set_xticklabels([g[0] for g in _grp_res], fontsize=fontsize_x_ticks)

    else:
        _gname, _mres = _grp_res[0]
        _x_cents = np.arange(len(_METHODS), dtype=float)

        for _mi, _m in enumerate(_METHODS):
            _col = _METHOD_COLORS[_m]
            _bm, _bs, _cm, _cs, _bns, _cns = _mres[_m]

            _bx = _x_cents[_mi] - _METAL_BAR_W / 2
            _cx = _x_cents[_mi] + _METAL_BAR_W / 2

            ax.bar(
                _bx,
                _bm,
                _METAL_BAR_W,
                yerr=_bs,
                capsize=2,
                color=_col,
                alpha=0.30,
                error_kw={"elinewidth": 0.8},
            )
            _add_n_label(ax, _bx, _bm, _bs, _bns, _col, single=True)

            ax.bar(
                _cx,
                _cm,
                _METAL_BAR_W,
                yerr=_cs,
                capsize=2,
                color=_col,
                alpha=1.0,
                error_kw={"elinewidth": 0.8},
            )
            _add_n_label(ax, _cx, _cm, _cs, _cns, _col)

        _panel_n_max = max(n for _m in _METHODS for n in _mres[_m][4] + _mres[_m][5])

        _xpad = 0.5
        ax.set_xlim(_x_cents[0] - _xpad, _x_cents[-1] + _xpad)
        ax.set_xticks(_x_cents)
        ax.set_xticklabels(
            [_display_method_name(m) for m in _METHODS],
            fontsize=fontsize_x_ticks,
        )

    if _pi in (0, 3):
        ax.set_ylabel("MBG Success Rate (%)", fontsize=fontsize_y_label)
    ax.set_ylim(0, 105)
    if _pi == 0:
        ax.set_title(_pspec["title"], fontsize=fontsize_titles, loc="left")
    else:
        ax.set_title(
            f"{_pspec['title']} (n$_{{\\mathrm{{max}}}}$ = {_panel_n_max})",
            fontsize=fontsize_titles,
            loc="left",
        )
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)

_all_h, _all_l = _axs[0].get_legend_handles_labels()
_by_lbl = dict(zip(_all_l, _all_h))

_chel_lbls = [l for l in _by_lbl if l.startswith("CHELATE")]
_base_lbls = [l for l in _by_lbl if "baseline" in l]

fig_aa.tight_layout()
fig_aa.subplots_adjust(wspace=0.38, hspace=0.30)

fig_aa.legend(
    [_by_lbl[l] for l in _chel_lbls],
    [l.replace("CHELATE (", "").rstrip(")") for l in _chel_lbls],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=3,
    fontsize=fontsize_legend,
    frameon=False,
    title="CHELATE trained on:",
    title_fontsize=fontsize_legend,
)
fig_aa.legend(
    [_by_lbl[l] for l in _base_lbls],
    [l.replace(" baseline", "") for l in _base_lbls],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=3,
    fontsize=fontsize_legend,
    frameon=False,
    title="Baselines",
    title_fontsize=fontsize_legend,
)
savefig_padded(
    fig_aa,
    os.path.join(output_folder, "EC_metal_subset_analysis.pdf"),
)
plt.show()

Top-5 metals in test set: ['ZN', 'MG', 'FE', 'CA', 'NI']
Metal counts: {'ZN': 45, 'MG': 19, 'CA': 15, 'FE': 12, 'NI': 10, 'CO': 7}


In [8]:
scatter_df = prediction_quality_inputs.get("scatter", pd.DataFrame())

from matplotlib.colors import LinearSegmentedColormap

fig, axes = plt.subplots(1, 2, figsize=figsize_two_panel, sharey=False)
dock_labels = {"redocked": "Re-docked", "crossdocked": "Cross-docked"}
hexbin_cmap = LinearSegmentedColormap.from_list("vina_seq", ["#FFFFFF", vina_color])
hbs = []

for ax, dock_key in zip(axes, ["redocked", "crossdocked"]):
    sub = (
        scatter_df[
            (scatter_df["dock_key"] == dock_key) & (scatter_df["method"] == "VINA")
        ]
        if not scatter_df.empty
        else pd.DataFrame()
    )
    if sub.empty:
        ax.set_visible(False)
        continue
    x = pd.to_numeric(sub["true_mbg_rmsd"], errors="coerce")
    y = pd.to_numeric(sub["pred"], errors="coerce")
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) == 0:
        ax.set_visible(False)
        continue
    r_val, _ = stats.pearsonr(x, y)
    lim_max = max(1.0, min(6.0, float(np.nanmax([x.max(), y.max()]))))
    hb = ax.hexbin(
        x,
        y,
        gridsize=40,
        extent=(0, lim_max, 0.0, 3.0),
        cmap=hexbin_cmap,
        mincnt=1,
        linewidths=0.2,
    )
    hbs.append(hb)
    ax.set_xlim(0, lim_max)
    ax.set_ylim(0.0, 3.0)
    ax.set_xlabel("True MBG RMSD (Å)", fontsize=fontsize_x_label)
    if dock_key == "redocked":
        ax.set_ylabel("Predicted MBG RMSD (Å)", fontsize=fontsize_y_label)
    ax.set_title(f"{dock_labels[dock_key]} (r={r_val:.3f})", fontsize=fontsize_titles)
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)

vmax = max(hb.get_array().max() for hb in hbs)
for hb in hbs:
    hb.set_clim(0, vmax)

fig.tight_layout()
cb = fig.colorbar(hbs[0], ax=axes, location="right", pad=0.02)
cb.set_label("Count", fontsize=fontsize_y_label)
cb.ax.tick_params(labelsize=fontsize_y_ticks)
savefig_padded(fig, os.path.join(output_folder, "scatter_pred_vs_true_mbg_rmsd.pdf"))
plt.show()

In [9]:
methods = ["VINA", "SMINA", "GNINA"]
method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}
dock_labels = {"redocked": "Re-docked", "crossdocked": "Cross-docked"}
cutoff_df = prediction_quality_inputs.get("cutoff", pd.DataFrame())

fig, axes = plt.subplots(1, 2, figsize=figsize_two_panel, sharey=False)

for ax, dock_key in zip(axes, ["redocked", "crossdocked"]):
    sub_df = (
        cutoff_df[cutoff_df["dock_key"] == dock_key].copy()
        if not cutoff_df.empty
        else pd.DataFrame()
    )
    if sub_df.empty:
        ax.set_visible(False)
        continue
    grp = sub_df.groupby(["method", "rmsd_cutoff"], as_index=False).agg(
        avg_poses_kept_mean=("avg_poses_kept", "mean"),
        avg_poses_kept_std=("avg_poses_kept", "std"),
        avg_poses_all_mean=("avg_poses_all", "mean"),
        avg_poses_all_std=("avg_poses_all", "std"),
    )
    grp["avg_poses_kept_std"] = grp["avg_poses_kept_std"].fillna(0.0)
    grp["avg_poses_all_std"] = grp["avg_poses_all_std"].fillna(0.0)
    for method in methods:
        subm = grp[grp["method"] == method].sort_values("rmsd_cutoff")
        if subm.empty:
            continue
        ax.errorbar(
            subm["rmsd_cutoff"],
            subm["avg_poses_kept_mean"],
            yerr=subm["avg_poses_kept_std"],
            marker="o",
            linestyle="-",
            capsize=3,
            color=method_colors[method],
            label=f"CHELATE trained on {_display_method_name(method)}",
        )
        base_mean = float(subm["avg_poses_all_mean"].iloc[0])
        base_std = float(subm["avg_poses_all_std"].iloc[0])
        ax.axhline(
            base_mean,
            color=method_colors[method],
            linestyle=":",
            linewidth=1.2,
            label=f"{_display_method_name(method)} pre-filter average",
        )
        ax.fill_between(
            subm["rmsd_cutoff"],
            base_mean - base_std,
            base_mean + base_std,
            color=method_colors[method],
            alpha=0.10,
        )
    ax.set_xlabel("Predicted MBG RMSD Cutoff (Å)", fontsize=fontsize_x_label)
    ax.set_title(dock_labels[dock_key], fontsize=fontsize_titles)
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)

axes[0].set_ylabel("Number of Kept Poses", fontsize=fontsize_y_label)

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)
by_label = {}
for h, l in zip(handles, labels):
    if l not in by_label:
        by_label[l] = h

chelate_labels = [l for l in by_label if l.startswith("CHELATE")]
avg_labels = [l for l in by_label if not l.startswith("CHELATE")]
chelate_handles = [by_label[l] for l in chelate_labels]
avg_handles = [by_label[l] for l in avg_labels]
chelate_short_labels = [l.replace("CHELATE trained on ", "") for l in chelate_labels]
avg_labels = [l.replace(" pre-filter average", "") for l in avg_labels]

fig.tight_layout()
fig.legend(
    chelate_handles,
    chelate_short_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=3,
    fontsize=fontsize_legend,
    frameon=False,
    title="CHELATE trained on:",
    title_fontsize=fontsize_legend,
)
fig.legend(
    avg_handles,
    avg_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=3,
    title="Baselines",
    title_fontsize=fontsize_legend,
    fontsize=fontsize_legend,
    frameon=False,
)
savefig_padded(fig, os.path.join(output_folder, "poses_kept_vs_cutoff.pdf"))
plt.show()

In [10]:
methods = ["VINA", "SMINA", "GNINA"]
method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}
dock_labels = {"redocked": "Re-docked", "crossdocked": "Cross-docked"}
all_grouped_pct = _all_grouped_sr_to_percent(all_grouped)
ext = _external_baselines_sr_to_percent(external_baselines)

fig, axes = plt.subplots(1, 2, figsize=figsize_two_panel, sharey=False)

for ax, dock_key in zip(axes, ["redocked", "crossdocked"]):
    x_ref = None
    for method in methods:
        grp = all_grouped_pct[method][dock_key]
        x = grp.index.values
        x_ref = x
        ax.errorbar(
            x,
            grp[("systems_count", "mean")],
            yerr=grp[("systems_count", "std")],
            marker="o",
            linestyle="-",
            capsize=3,
            color=method_colors[method],
            label=f"CHELATE trained on {_display_method_name(method)}",
        )
    if x_ref is not None:
        # All methods are compared on the same test pairs (the common cohort).
        ax.axhline(
            all_grouped_pct[methods[0]][dock_key][("total_systems", "mean")].iloc[0],
            color="grey",
            linestyle="-.",
            linewidth=1.2,
        )
        # True MBG coverage: systems retained when using true MBG RMSD as cutoff
        _cdf = prediction_quality_inputs["cutoff"]
        if not _cdf.empty and x_ref is not None:
            _true_grp = (
                _cdf[_cdf["dock_key"] == dock_key]
                .groupby(["method", "rmsd_cutoff"], as_index=False)
                .agg(
                    true_cov_mean=("true_mbg_coverage", "mean"),
                    true_cov_std=("true_mbg_coverage", "std"),
                )
            )
            _true_grp["true_cov_std"] = _true_grp["true_cov_std"].fillna(0.0)
            for method in methods:
                _sub = _true_grp[_true_grp["method"] == method].sort_values(
                    "rmsd_cutoff"
                )
                if _sub.empty:
                    continue
                ax.errorbar(
                    _sub["rmsd_cutoff"],
                    _sub["true_cov_mean"],
                    yerr=_sub["true_cov_std"],
                    marker="",
                    linestyle="-.",
                    linewidth=1.2,
                    capsize=2,
                    color=method_colors[method],
                    label=f"{_display_method_name(method)} true MBG baseline",
                )
        ax.set_xlim(float(np.min(x_ref)), float(np.max(x_ref)))
    ax.set_xlabel("Predicted MBG RMSD Cutoff (Å)", fontsize=fontsize_x_label)
    ax.set_title(dock_labels[dock_key], fontsize=fontsize_titles)
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)

axes[0].set_ylabel("Number of Systems", fontsize=fontsize_y_label)

handles, labels = [], []
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    handles.extend(h)
    labels.extend(l)
by_label = {}
for h, l in zip(handles, labels):
    if l not in by_label:
        by_label[l] = h

chelate_labels = [l for l in by_label if l.startswith("CHELATE")]
ideal_labels = [l for l in by_label if l.endswith("true MBG baseline")]
chelate_handles = [by_label[l] for l in chelate_labels]
ideal_handles = [by_label[l] for l in ideal_labels]
chelate_short_labels = [l.replace("CHELATE trained on ", "") for l in chelate_labels]

fig.tight_layout()
fig.legend(
    chelate_handles,
    chelate_short_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.04),
    ncol=3,
    fontsize=fontsize_legend,
    frameon=False,
    title="CHELATE trained on:",
    title_fontsize=fontsize_legend,
)
ideal_short_labels = [l.replace(" true MBG baseline", "") for l in ideal_labels]
fig.legend(
    ideal_handles,
    ideal_short_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.06),
    ncol=3,
    title="Ideal",
    title_fontsize=fontsize_legend,
    fontsize=fontsize_legend,
    frameon=False,
)
savefig_padded(fig, os.path.join(output_folder, "system_coverage_vs_cutoff.pdf"))
plt.show()

In [11]:
def _plot_fraction_panel(ax, grouped, baseline_df, column, dock_key, method_colors):
    for method in ["VINA", "SMINA", "GNINA"]:
        sub = grouped[
            (grouped["method"] == method) & (grouped["dock_key"] == dock_key)
        ].sort_values("fraction")
        if sub.empty:
            continue
        x_ref = sub["fraction"].values
        ax.errorbar(
            sub["fraction"],
            sub[f"{column}_mean"],
            yerr=sub[f"{column}_std"],
            marker="o",
            linestyle="-",
            capsize=3,
            color=method_colors[method],
            label=f"CHELATE trained on {_display_method_name(method)}",
        )
        if baseline_df is None:
            continue
        bsub = baseline_df[
            (baseline_df["method"] == method) & (baseline_df["dock_key"] == dock_key)
        ]
        if not bsub.empty:
            b_mean = float(bsub["base_mean"].iloc[0])
            b_std = float(bsub["base_std"].iloc[0])
            ax.axhline(
                b_mean,
                color=method_colors[method],
                linestyle=":",
                linewidth=1.2,
                label=f"{_display_method_name(method)} baseline",
            )
            ax.fill_between(
                x_ref,
                b_mean - b_std,
                b_mean + b_std,
                color=method_colors[method],
                alpha=0.12,
            )


def _plot_mbg_sr_vs_training_fraction(fraction_df, output_folder, cutoff_tag, n_pairs):
    """Success among retained systems, retained fraction and success over all systems
    (rejected systems count as failures) against the training fraction."""
    method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}
    dock_labels = {"redocked": "Re-docked", "crossdocked": "Cross-docked"}

    fraction_df = fraction_df.assign(
        retained=100.0 * fraction_df["systems_count"] / n_pairs,
        mbg_sr_all=fraction_df["mbg_sr_top1"].fillna(0.0)
        * fraction_df["systems_count"]
        / n_pairs,
    )
    grouped = fraction_df.groupby(
        ["method", "dock_key", "fraction"], as_index=False
    ).agg(
        mbg_sr_mean=("mbg_sr_top1", "mean"),
        mbg_sr_std=("mbg_sr_top1", "std"),
        retained_mean=("retained", "mean"),
        retained_std=("retained", "std"),
        mbg_sr_all_mean=("mbg_sr_all", "mean"),
        mbg_sr_all_std=("mbg_sr_all", "std"),
    )
    for column in ("mbg_sr", "mbg_sr_all"):
        grouped[f"{column}_std"] = grouped[f"{column}_std"].fillna(0.0) * 100.0
        grouped[f"{column}_mean"] = grouped[f"{column}_mean"] * 100.0
    grouped["retained_std"] = grouped["retained_std"].fillna(0.0)

    baseline_df = (
        fraction_df[["method", "dock_key", "rep_id", "baseline_mbg_sr_top1"]]
        .drop_duplicates()
        .groupby(["method", "dock_key"], as_index=False)
        .agg(
            base_mean=("baseline_mbg_sr_top1", "mean"),
            base_std=("baseline_mbg_sr_top1", "std"),
        )
    )
    baseline_df["base_mean"] = baseline_df["base_mean"] * 100.0
    baseline_df["base_std"] = baseline_df["base_std"].fillna(0.0) * 100.0

    # Rows: success among retained systems, retained fraction, success over all
    # systems. The docking baseline is drawn where it is comparable (rows 1 and 3).
    rows = [
        ("mbg_sr", "MBG success,\nretained (%)", True, (0.0, 80.0)),
        ("retained", "Systems\nretained (%)", False, (20.0, 100.0)),
        ("mbg_sr_all", "MBG success,\nall systems (%)", True, (0.0, 80.0)),
    ]
    fig, axes_grid = plt.subplots(
        len(rows), 2, figsize=(16 * cm, 17 * cm), sharex=True, sharey="row"
    )
    for (column, ylabel, show_baseline, ylim), axes in zip(rows, axes_grid):
        for ax, dock_key in zip(axes, ["redocked", "crossdocked"]):
            _plot_fraction_panel(
                ax,
                grouped,
                baseline_df if show_baseline else None,
                column,
                dock_key,
                method_colors,
            )
            ax.grid(alpha=0.3)
            ax.set_xlim(0.08, 1.02)
            ax.set_ylim(*ylim)
            ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
            ax.tick_params(axis="y", labelsize=fontsize_y_ticks)
        axes[0].set_ylabel(ylabel, fontsize=fontsize_y_label)
    for ax, dock_key in zip(axes_grid[0], ["redocked", "crossdocked"]):
        ax.set_title(dock_labels[dock_key], fontsize=fontsize_titles)
    for ax in axes_grid[-1]:
        ax.set_xlabel("Data Fraction", fontsize=fontsize_x_label)
    axes = axes_grid.ravel()

    handles, labels = [], []
    for ax in axes:
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
    by_label = {}
    for h, l in zip(handles, labels):
        if l not in by_label:
            by_label[l] = h

    chelate_labels = [l for l in by_label if l.startswith("CHELATE")]
    baseline_labels = [l for l in by_label if not l.startswith("CHELATE")]
    chelate_handles = [by_label[l] for l in chelate_labels]
    baseline_handles = [by_label[l] for l in baseline_labels]
    chelate_short_labels = [
        l.replace("CHELATE trained on ", "") for l in chelate_labels
    ]
    baseline_short_labels = [l.replace(" baseline", "") for l in baseline_labels]

    fig.tight_layout()
    fig.legend(
        chelate_handles,
        chelate_short_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=3,
        fontsize=fontsize_legend,
        frameon=False,
        title="CHELATE trained on:",
        title_fontsize=fontsize_legend,
    )
    fig.legend(
        baseline_handles,
        baseline_short_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=3,
        title="Baselines",
        title_fontsize=fontsize_legend,
        fontsize=fontsize_legend,
        frameon=False,
    )
    savefig_padded(
        fig,
        os.path.join(
            output_folder,
            f"mbg_sr_top1_vs_training_fraction_predCutoff={cutoff_tag}.pdf",
        ),
    )
    plt.show()


# The sweep at the selected cutoffs; the file name encodes them (cutoff_map_tag in
# model/prediction_cutoffs.py), e.g. VINA-0p7_SMINA-0p8_GNINA-0p85.
_sweep_tag = "_".join(
    f"{m}-{PRED_CUTOFFS[m]:g}".replace(".", "p") for m in ("VINA", "SMINA", "GNINA")
)
_sweep_csv = os.path.join(
    csv_folder, f"mbg_sr_vs_training_fraction_predCutoff={_sweep_tag}.csv"
)
# Denominator for the retained fraction and all-systems success: the common cohort.
_n_pairs = len(pd.read_csv("../../docking/cohorts/comparison_common_test.csv"))
_plot_mbg_sr_vs_training_fraction(
    pd.read_csv(_sweep_csv), output_folder, _sweep_tag, _n_pairs
)

In [12]:
methods = ["VINA", "SMINA", "GNINA"]
method_colors = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}
top_n = 5

fi_dfs = [
    pd.read_csv(p) for p in sorted(glob(os.path.join(csv_folder, "interpret_fi_*.csv")))
]
shap_dfs = [
    pd.read_csv(p)
    for p in sorted(glob(os.path.join(csv_folder, "interpret_shap_*.csv")))
]
fi_all = pd.concat(fi_dfs, ignore_index=True) if fi_dfs else pd.DataFrame()
shap_all = pd.concat(shap_dfs, ignore_index=True) if shap_dfs else pd.DataFrame()

methods_present = [
    m for m in methods if not fi_all.empty and m in fi_all["method"].values
]
if not methods_present:
    print("No interpretability CSVs found — run final_model_run.py first.")
else:
    n_methods = len(methods_present)

    def _plot_interp(all_df, value_col, xlabel, fname, cmap_fn, hide_yticks=False):
        fig, axes = plt.subplots(1, n_methods, figsize=(16 * cm, 8 * cm))
        if n_methods == 1:
            axes = [axes]
        for idx, (ax, method) in enumerate(zip(axes, methods_present)):
            sub = (
                all_df[all_df["method"] == method]
                .sort_values(
                    "mean_abs_shap" if "shap" in value_col else "importance",
                    ascending=False,
                )
                .head(top_n)
            )
            sub = sub.sort_values(value_col, ascending=False)
            colors = cmap_fn(sub[value_col], method)
            ax.bar(sub["feature"], sub[value_col], color=colors)
            ax.set_ylabel(xlabel, fontsize=fontsize_y_label)
            ax.set_title(
                f"{'abc'[idx]}) {_display_method_name(method)}",
                fontsize=fontsize_titles,
                loc="left",
            )
            ax.grid(axis="y", alpha=0.25)
            ax.tick_params(
                axis="x", labelsize=max(fontsize_x_ticks - 1, 7), rotation=90
            )
            ax.tick_params(axis="y", labelsize=max(fontsize_y_ticks - 1, 7))
            if hide_yticks:
                ax.tick_params(axis="y", left=False, labelleft=False)
        fig.tight_layout()
        savefig_padded(fig, os.path.join(output_folder, fname))
        plt.show()

    _plot_interp(
        fi_all,
        "importance",
        xlabel="Prediction Values Change",
        fname="interpret_feature_importance.pdf",
        cmap_fn=lambda vals, method: method_colors[method],
        hide_yticks=True,
    )
    _plot_interp(
        shap_all,
        "mean_abs_shap",
        xlabel="Mean |SHAP value|",
        fname="interpret_shap_importance.pdf",
        cmap_fn=lambda vals, method: method_colors[method],
        hide_yticks=True,
    )

## CHELATE as a selective filter

Precision-coverage curves and the decomposition of the gain among retained systems into system selection and pose filtering (Fig. 4).


In [13]:
# CHELATE as a selective filter (1/2): precision-coverage curves.
# x = retained systems (%), y = top-1 MBG success among retained systems (%), one point per
# prediction cutoff (mean ± SD over the three docking runs). Docking programs without
# filtering sit at 100 % coverage. If csv_files/distance_baseline_test_curve.csv exists
# (test_curve.csv from `python -m model.distance_baseline`), the distance-cutoff filter
# is drawn for comparison.
_SEL_METHODS = ["VINA", "SMINA", "GNINA"]
_SEL_COLORS = {"VINA": vina_color, "SMINA": smina_color, "GNINA": gnina_color}
_SR = f"MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}"
_BASE = f"baseline_MBG_sr_top1_RMSD={MBG_RMSD_THRESHOLD:g}"
_grouped_flat = pd.read_csv(os.path.join(csv_folder, "plot_repro_grouped_metrics.csv"))
_ext_flat = pd.read_csv(os.path.join(csv_folder, "plot_repro_external_baselines.csv"))
_dist_path = os.path.join(csv_folder, "distance_baseline_test_curve.csv")
_dist = pd.read_csv(_dist_path) if os.path.exists(_dist_path) else None
if _dist is None:
    print(
        "No distance-baseline curve found; plotting CHELATE and docking baselines only."
    )

fig, axes = plt.subplots(1, 2, figsize=figsize_two_panel, sharey=True)
for ax, dock in zip(axes, ["crossdocked", "redocked"]):
    for method in _SEL_METHODS:
        g = _grouped_flat[
            (_grouped_flat["method"] == method) & (_grouped_flat["dock_key"] == dock)
        ].sort_values("rmsd_cutoff")
        cov = 100 * g["systems_count__mean"] / g["total_systems__mean"]
        ax.errorbar(
            cov,
            100 * g[f"{_SR}__mean"],
            yerr=100 * g[f"{_SR}__std"],
            color=_SEL_COLORS[method],
            marker="o",
            markersize=3,
            linewidth=1,
            capsize=2,
        )
        op = g[np.isclose(g["rmsd_cutoff"], PRED_CUTOFFS[method])]
        ax.scatter(
            100 * op["systems_count__mean"] / op["total_systems__mean"],
            100 * op[f"{_SR}__mean"],
            s=60,
            marker="*",
            color=_SEL_COLORS[method],
            edgecolor="black",
            linewidth=0.5,
            zorder=5,
        )
        base = g.iloc[-1]
        ax.scatter(
            100,
            100 * base[f"{_BASE}__mean"],
            s=30,
            marker="s",
            color=_SEL_COLORS[method],
            edgecolor="black",
            linewidth=0.5,
            zorder=5,
        )
        if _dist is not None:
            d = _dist[(_dist["method"] == method) & (_dist["dock"] == dock)]
            d = d.groupby("d")[["retained_fraction", "sr_retained"]].mean()
            ax.plot(
                100 * d["retained_fraction"],
                100 * d["sr_retained"],
                color=_SEL_COLORS[method],
                linestyle="--",
                linewidth=1,
            )
    for method in EXTRA_BASELINE_METHODS:
        e = _ext_flat[
            (_ext_flat["dock_key"] == dock)
            & (_ext_flat["method"] == method)
            & (_ext_flat["metric_family"] == "mbg")
            & np.isclose(_ext_flat["threshold"].astype(float), MBG_RMSD_THRESHOLD)
            & (_ext_flat["top_k"] == 1)
        ]
        if len(e):
            ax.scatter(
                100,
                100 * e["mean"].iloc[0],
                s=30,
                marker="D",
                color=EXTRA_BASELINE_METHOD_COLORS[method],
                edgecolor="black",
                linewidth=0.5,
                zorder=5,
            )
    ax.set_title(
        "Cross-docked" if dock == "crossdocked" else "Re-docked",
        fontsize=fontsize_titles,
    )
    ax.set_xlabel("Retained systems (%)", fontsize=fontsize_x_label)
    ax.set_xlim(0, 105)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=fontsize_x_ticks)
axes[0].set_ylabel(
    "Top-1 MBG success rate,\nretained systems (%)", fontsize=fontsize_y_label
)

_handles = [
    Line2D([0], [0], color=_SEL_COLORS[m], marker="o", markersize=3)
    for m in _SEL_METHODS
]
_labels = [f"CHELATE ({_display_method_name(m)})" for m in _SEL_METHODS]
_handles += [
    Line2D(
        [0],
        [0],
        color="grey",
        marker="*",
        markersize=8,
        markeredgecolor="black",
        linestyle="",
    ),
    Line2D(
        [0],
        [0],
        color="grey",
        marker="s",
        markersize=5,
        markeredgecolor="black",
        linestyle="",
    ),
]
_labels += ["Selected cutoff", "Docking program, no filter"]
_handles += [
    Line2D(
        [0],
        [0],
        color=EXTRA_BASELINE_METHOD_COLORS[m],
        marker="D",
        markersize=5,
        markeredgecolor="black",
        linestyle="",
    )
    for m in EXTRA_BASELINE_METHODS
]
_labels += [_display_method_name(m) for m in EXTRA_BASELINE_METHODS]
if _dist is not None:
    _handles.append(Line2D([0], [0], color="grey", linestyle="--"))
    _labels.append("Distance-cutoff filter")
fig.tight_layout()
legend_below_axes(fig, axes, _handles, _labels, ncol=3, fontsize=fontsize_legend)
savefig_padded(fig, os.path.join(output_folder, "selective_precision_coverage.pdf"))
plt.show()

In [14]:
# CHELATE as a selective filter (2/2): where the gain among retained systems comes from.
# For each docking run at the selected cutoff: the docking program's top-1 success over all
# systems, on the systems CHELATE retains and on those it discards, and CHELATE's pick on
# the retained systems. The difference between bars 1 and 2 is the effect of selecting
# systems; between bars 2 and 3, the effect of filtering poses within a system.
# Mean ± SD over the three docking runs.
_scatter_all = prediction_quality_inputs["scatter"]
_rows = []
for dock in ["crossdocked", "redocked"]:
    for method in _SEL_METHODS:
        sub = _scatter_all[
            (_scatter_all["method"] == method) & (_scatter_all["dock_key"] == dock)
        ]
        _PRED_CUT = PRED_CUTOFFS[method]
        for rep, r in sub.groupby("rep_id"):
            first = r.groupby("system_id__ligand_id")["pose"].transform("min")
            top1 = r[r["pose"] == first].set_index("system_id__ligand_id")
            ok = top1["true_mbg_rmsd"] <= MBG_RMSD_THRESHOLD
            kept_ids = set(r.loc[r["pred"] <= _PRED_CUT, "system_id__ligand_id"])
            kept = top1.index.isin(kept_ids)
            pick = (
                r[r["pred"] <= _PRED_CUT]
                .sort_values("pose")
                .groupby("system_id__ligand_id")
                .first()
            )
            best = (
                r.groupby("system_id__ligand_id")["true_mbg_rmsd"].min()
                <= MBG_RMSD_THRESHOLD
            )
            in_kept = best.index.isin(kept_ids)
            _rows.append(
                {
                    "dock": dock,
                    "method": method,
                    "rep": rep,
                    "retained_fraction": kept.mean(),
                    "docking_all": ok.mean(),
                    "docking_retained": ok[kept].mean(),
                    "chelate_retained": (
                        pick["true_mbg_rmsd"] <= MBG_RMSD_THRESHOLD
                    ).mean(),
                    "docking_discarded": ok[~kept].mean(),
                    "sampled_retained": best[in_kept].mean(),
                    "sampled_discarded": best[~in_kept].mean(),
                }
            )
_decomp = pd.DataFrame(_rows)
_decomp_summary = _decomp.groupby(["dock", "method"]).agg(["mean", "std"])
print((100 * _decomp_summary.drop(columns="rep")).round(1).to_string())

_BARS = [
    ("docking_all", "Docking top-1, all systems"),
    ("docking_retained", "Docking top-1, retained systems"),
    ("chelate_retained", "CHELATE, retained systems"),
    ("docking_discarded", "Docking top-1, discarded systems"),
]
fig, axes = plt.subplots(1, 2, figsize=figsize_two_panel, sharey=True)
_w = 0.2
for ax, dock in zip(axes, ["crossdocked", "redocked"]):
    for i, method in enumerate(_SEL_METHODS):
        s = _decomp_summary.loc[(dock, method)]
        col = _SEL_COLORS[method]
        styles = [
            {"color": "lightgrey", "edgecolor": "black"},
            {"color": _lighten_color(col, 0.55), "edgecolor": "black"},
            {"color": col, "edgecolor": "black"},
            {"color": "white", "edgecolor": col, "hatch": "////"},
        ]
        for j, ((key, _), style) in enumerate(zip(_BARS, styles)):
            ax.bar(
                i + (j - 1.5) * _w,
                100 * s[(key, "mean")],
                width=_w,
                yerr=100 * s[(key, "std")],
                capsize=2,
                linewidth=0.6,
                error_kw={"linewidth": 0.8},
                **style,
            )
        ax.text(
            i,
            -9,
            f"{100 * s[('retained_fraction', 'mean')]:.0f}% retained",
            ha="center",
            fontsize=fontsize_legend - 1,
        )
    ax.set_xticks(range(len(_SEL_METHODS)))
    ax.set_xticklabels(
        [_display_method_name(m) for m in _SEL_METHODS], fontsize=fontsize_x_ticks
    )
    ax.set_title(
        "Cross-docked" if dock == "crossdocked" else "Re-docked",
        fontsize=fontsize_titles,
    )
    ax.set_ylim(0, 60)
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(labelsize=fontsize_y_ticks)
axes[0].set_ylabel("Top-1 MBG success rate (%)", fontsize=fontsize_y_label)
_leg_styles = [
    {"facecolor": "lightgrey", "edgecolor": "black"},
    {"facecolor": _lighten_color(vina_color, 0.55), "edgecolor": "black"},
    {"facecolor": vina_color, "edgecolor": "black"},
    {"facecolor": "white", "edgecolor": vina_color, "hatch": "////"},
]
from matplotlib.patches import Patch

fig.tight_layout()
legend_below_axes(
    fig,
    axes,
    [Patch(**st) for st in _leg_styles],
    [lbl for _, lbl in _BARS],
    ncol=2,
    fontsize=fontsize_legend,
)
savefig_padded(fig, os.path.join(output_folder, "selective_gain_decomposition.pdf"))
plt.show()

                   retained_fraction      docking_all      docking_retained      chelate_retained      docking_discarded      sampled_retained      sampled_discarded     
                                mean  std        mean  std             mean  std             mean  std              mean  std             mean  std              mean  std
dock        method                                                                                                                                                        
crossdocked GNINA               59.0  1.8        18.1  2.1             24.4  4.0             40.2  1.0               9.0  1.6             64.1  1.6              39.8  6.4
            SMINA               61.6  2.0        18.4  1.3             22.5  2.1             34.0  1.6              11.7  2.8             60.6  4.6              39.7  3.4
            VINA                57.9  2.1        18.1  0.5             23.5  2.1             32.2  2.7              10.8  2.5             61.5  3

Figs. S25 and S27 (metal-identity probe) are made by `../metal_identity/make_metal_identity_figures.py`.
